In [1]:
import socket

In [2]:
import threading

In [3]:
import time

# localhost — server only accepts connections from this same machine

In [4]:
HOST = "127.0.0.1"

# the "door number" clients must connect to

In [5]:
PORT = 65002

# This is the function that runs inside its own thread, once per connected client:

In [6]:
def handle_client(conn, addr):
    """This is the WORKER: does the actual work for one client."""
    with conn:
        # threading.get_ident():=> Return the ‘thread identifier’ of the current thread. This is a nonzero integer. 
        # Its value has no direct meaning; it is intended as a magic cookie to be used e.g. 
        # to index a dictionary of thread-specific data. Thread identifiers may be recycled 
        # when a thread exits and another thread is created.
        
        print(f"[WORKER-{threading.get_ident()}] Handling {addr}") 
        
        # read up to 1024 bytes from the client, decode bytes → string
        request = conn.recv(1024).decode() 

        # Simulate a slow operation (e.g. a DB query) so concurrency
        # is easy to observe
        # Suspend execution of the calling thread for the given number of seconds. 
        # The argument may be a floating-point number to indicate a more precise sleep time.
        time.sleep(2)

        reply = f"Processed '{request}' by worker thread {threading.get_ident()}"
        conn.sendall(reply.encode())
        print(f"[WORKER-{threading.get_ident()}] Done with {addr}")

# The Dispatcher — main()

# socket.AF_INET = use IPv4

# socket.SOCK_STREAM = use TCP

# socket.SOL_SOCKET = It is a constant in Python's socket module that specifies the socket API layer itself.

# socket.SO_REUSEADDR = lets you restart the server quickly without getting a "port already in use" error.

# listen(5) = sets how many pending connections the OS will queue up before rejecting new ones.

# worker.start() = Starting a thread does not wait for that thread to finish — it just launches it in the background and returns control immediately.

In [7]:
def main():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
        server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server_socket.bind((HOST, PORT))
        server_socket.listen(5)
        print(f"[DISPATCHER] Listening on {HOST}:{PORT}")

        while True:
            conn, addr = server_socket.accept()
            print(f"[DISPATCHER] Accepted {addr}, spawning worker thread")
            # Hand off to a worker thread and go straight back to accept()
            worker = threading.Thread(target=handle_client, args=(conn, addr))
            worker.start()

# Each new client that connects gets its own brand-new threading.Thread, so if 5 clients connect nearly simultaneously, you end up with 5 worker threads all running time.sleep(2) at the same time, not one after another.

In [8]:
if __name__ == "__main__":
    main()

[DISPATCHER] Listening on 127.0.0.1:65002
[DISPATCHER] Accepted ('127.0.0.1', 52274), spawning worker thread
[WORKER-15420] Handling ('127.0.0.1', 52274)
[DISPATCHER] Accepted ('127.0.0.1', 52275), spawning worker thread
[WORKER-28820] Handling ('127.0.0.1', 52275)
[DISPATCHER] Accepted ('127.0.0.1', 52276), spawning worker thread
[WORKER-27020] Handling ('127.0.0.1', 52276)
[DISPATCHER] Accepted ('127.0.0.1', 52277), spawning worker thread
[WORKER-1128] Handling ('127.0.0.1', 52277)
[DISPATCHER] Accepted ('127.0.0.1', 52278), spawning worker thread
[WORKER-19208] Handling ('127.0.0.1', 52278)
[DISPATCHER] Accepted ('127.0.0.1', 52294), spawning worker thread
[WORKER-2892] Handling ('127.0.0.1', 52294)
[DISPATCHER] Accepted ('127.0.0.1', 52282), spawning worker thread
[WORKER-15580] Handling ('127.0.0.1', 52282)
[DISPATCHER] Accepted ('127.0.0.1', 52285), spawning worker thread
[WORKER-17736] Handling ('127.0.0.1', 52285)
[DISPATCHER] Accepted ('127.0.0.1', 52287), spawning worker thre

: 